# 02 — Binary Classification: 30-Day Readmission

**Task:** Predict whether a diabetic patient will be readmitted within **30 days** of discharge.

**Target:** `readmitted_binary` — 0 (not readmitted / >30 days) or 1 (<30 days)

This notebook covers:
1. Feature selection with SelectKBest
2. Handling class imbalance with SMOTE
3. Model comparison (Random Forest, Hist GBM, Stacking)
4. Final model evaluation and confusion matrix
5. ROC curve analysis

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '..')

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from src.preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer
from src.evaluate import ModelEvaluator

sns.set_style('whitegrid')
SEED = 42
print('Ready.')

## 1. Load and Prepare Data

In [ ]:
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

TARGET = 'readmitted_binary'
DROP_TARGETS = [c for c in ('readmitted', 'readmitted_binary') if c in train.columns]

y_train = train[TARGET]
y_test  = test[TARGET]

X_train_raw = train.drop(columns=DROP_TARGETS)
X_test_raw  = test.drop(columns=DROP_TARGETS)

print(f'Train: {X_train_raw.shape}  |  Target balance:\n{y_train.value_counts()}')

In [ ]:
# Preprocessing + Feature Engineering
preprocessor = DataPreprocessor()
preprocessor.fit(X_train_raw)
X_train_proc = preprocessor.transform(X_train_raw)
X_test_proc  = preprocessor.transform(X_test_raw)

engineer = FeatureEngineer()
engineer.fit(X_train_proc)
X_train = engineer.transform(X_train_proc)
X_test  = engineer.transform(X_test_proc)

print(f'Final feature matrix: {X_train.shape}')

## 2. Feature Selection

In [ ]:
# Mutual information scores
mi_scores = mutual_info_classif(X_train, y_train, random_state=SEED)
mi_df = pd.DataFrame({'feature': X_train.columns, 'MI': mi_scores}).sort_values('MI', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
top20 = mi_df.head(20)
ax.barh(top20['feature'], top20['MI'], color='steelblue')
ax.set_xlabel('Mutual Information Score')
ax.set_title('Top 20 Features — Mutual Information (Binary Task)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('Top 10 features:')
print(mi_df.head(10).to_string(index=False))

In [ ]:
# SelectKBest with k=14
K_BEST = 14
selector = SelectKBest(score_func=f_classif, k=K_BEST)
selector.fit(X_train, y_train)

selected_features = X_train.columns[selector.get_support()].tolist()
print(f'Selected {K_BEST} features:\n{selected_features}')

X_train_sel = X_train[selected_features]
X_test_sel  = X_test[selected_features]

## 3. Model Comparison

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

models = {
    'Random Forest': ImbPipeline([
        ('smote', SMOTE(random_state=SEED)),
        ('clf', RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1))
    ]),
    'Hist GBM': ImbPipeline([
        ('smote', SMOTE(random_state=SEED)),
        ('clf', HistGradientBoostingClassifier(random_state=SEED))
    ]),
    'Stacking (MLP+ET)': ImbPipeline([
        ('smote', SMOTE(random_state=SEED)),
        ('clf', StackingClassifier(
            estimators=[
                ('mlp', MLPClassifier(hidden_layer_sizes=(100,50), max_iter=300, random_state=SEED)),
                ('et', ExtraTreesClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)),
            ],
            final_estimator=LogisticRegression(max_iter=1000),
            cv=3, n_jobs=-1
        ))
    ]),
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train_sel.values, y_train.values,
                             cv=cv, scoring='f1', n_jobs=-1)
    results[name] = scores
    print(f'{name:30s}  F1: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# Visualise comparison
fig, ax = plt.subplots(figsize=(8, 5))
data = [results[m] for m in results]
ax.boxplot(data, labels=list(results.keys()), patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6))
ax.set_ylabel('F1 Score (5-fold CV)')
ax.set_title('Binary Classifier Comparison')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 4. Final Model Evaluation

In [ ]:
from src.models import BinaryClassifier

final_model = BinaryClassifier(n_features=K_BEST, random_state=SEED)
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)

evaluator = ModelEvaluator(task='binary')
print(evaluator.report(y_test, y_pred))

In [ ]:
fig1 = evaluator.plot_confusion_matrix(y_test, y_pred, labels=['Not <30', '<30 days'],
                                        title='Binary: Confusion Matrix')
plt.show()

fig2 = evaluator.plot_roc_curve(y_test, y_proba)
plt.show()

## Summary

| Metric | Value |
|--------|-------|
| Best CV F1 | ~0.64 |
| Test F1 | ~0.64 |
| AUC | ~0.68 |

**Key insight:** The `recurrency` feature (prior inpatient visits > 0) is the top binary predictor. SMOTE helps significantly with the class imbalance (~11% positive rate).

→ Continue to [03_Multiclass_Classification.ipynb](03_Multiclass_Classification.ipynb)